# 1st Hidden Layer — Evaluate Little-Perturbation Checkpoints

In the 1st-layer perturbation sweeps, models trained with a *little*
perturbation tend to score better at their matched evaluation level. This
notebook isolates that effect: for each perturbation type it loads a **single
checkpoint trained at a small perturbation level** and evaluates that one model
across the **entire** perturbation sweep (eval-on-checkpoint), rather than
training a fresh model per level.

Each perturbation is evaluated twice — once for the **no-delay** model and once
for the **delay** model — using the same checkpoint level:

- **Jitter** — per-spike Gaussian jitter; default checkpoint `sigma = 5`.
- **Shift** — per-neuron Gaussian shift; default checkpoint `sigma = 5`.
- **Deletion** — per-spike deletion; default checkpoint `p_d = 0.2`.

Evaluations cover the **whole / part / norm** SHD variants. Per-section results
are written to `log_eval_on_perturbatedModel/` with a `1stLayer` tag so they
stay distinct from the 2nd-layer eval-on-checkpoint outputs.

In [12]:
# Setup: reuse the 1st-layer training modules (model classes, data pipeline
# and evaluation routines) so the evaluation matches the original sweeps.
import sys
import json
from pathlib import Path

import numpy as np
import torch

BASE_DIR = Path.cwd()
assert (BASE_DIR / "jitter").is_dir(), (
    "Run this notebook from my_project/code/perturbation/ so the "
    "jitter / shift / deletion packages are importable."
)

for sub in ("jitter", "shift", "deletion"):
    sub_path = str((BASE_DIR / sub).resolve())
    if sub_path not in sys.path:
        sys.path.append(sub_path)

import jitter_train as jitter_mod
import shift_train as shift_mod
import deletion_train as deletion_mod

device = jitter_mod.device
print(f"Device: {device}")

# Dataset variants to evaluate (matches result_visualization_1stLayer.ipynb).
EVAL_DATASETS = ("whole", "part", "norm")

# Destination for the eval-on-checkpoint sweep results.
EVAL_LOG_DIR = Path("log_eval_on_perturbatedModel")
EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Eval results dir: {EVAL_LOG_DIR.resolve()}")

Device: cuda
Eval results dir: D:\IC_2025\IRP\workspace\my_project\code\perturbation\log_eval_on_perturbatedModel


## Shared evaluation helpers

`DATASET_CONFIGS`, `SIM_PARAMS` and the split logic are identical across the
three training modules, so the helpers below read them from `jitter_mod`.
`test_with_repeats` takes the perturbation level as its third positional
argument (`sigma` for jitter/shift, `p_d` for deletion), so one evaluation
routine serves all three perturbation types and both delay modes.

The only difference from the 2nd-layer notebook is the checkpoint filename:
1st-layer checkpoints carry no `_2ndLayer` infix
(`{perturbation}_{dataset}_{delay}_{token}.pt`), while the output JSON is
tagged with `1stLayer` so both layers can share `EVAL_LOG_DIR`.

In [13]:
_TEST_LOADER_CACHE: dict[str, object] = {}


def get_test_loader(dataset_key: str):
    """Return the test DataLoader for ``dataset_key``.

    Uses the same split ranges and seed as training, so the evaluation set
    matches the one behind the original sweep results. Cached because the
    test data is identical across perturbation types and delay modes.
    """
    if dataset_key not in _TEST_LOADER_CACHE:
        cfg = jitter_mod.DATASET_CONFIGS[dataset_key]
        X, Y = jitter_mod.load_shd_data(
            cfg["mat_file"], target_T=jitter_mod.SIM_PARAMS["tSample"]
        )
        _, _, test_loader = jitter_mod.build_dataloaders(
            X, Y, batch_size=jitter_mod.BATCH_SIZE, seed=jitter_mod.SEED
        )
        _TEST_LOADER_CACHE[dataset_key] = test_loader
    return _TEST_LOADER_CACHE[dataset_key]


def load_checkpoint(module, net_class, dataset_key, ckpt_filename, use_delay):
    """Instantiate a network (delay or no-delay) and load a saved checkpoint."""
    cfg = module.DATASET_CONFIGS[dataset_key]
    net = net_class(
        input_dim=cfg["input_dim"],
        hidden_units=module.HIDDEN_UNITS,
        num_classes=module.NUM_CLASSES,
        use_delay=use_delay,
        max_delay=module.MAX_DELAY,
    ).to(module.device)
    ckpt_path = module.DATA_DIR / ckpt_filename
    state = torch.load(ckpt_path, map_location=module.device)
    net.load_state_dict(state)
    net.eval()
    return net


def evaluate_across_levels(module, net, dataset_key, eval_levels):
    """Evaluate one fixed model at every perturbation level."""
    test_loader = get_test_loader(dataset_key)
    results = {}
    for level in eval_levels:
        res = module.test_with_repeats(net, test_loader, level)
        results[level] = res
        print(f"      eval@{level}: {res['mean']:.4f} +/- {res['std']:.4f}")
    return results


def save_sweep_json(results, out_path, key_fn):
    """Serialise eval results to the same schema as the training sweeps."""
    serial = {
        key_fn(level): {
            "mean": float(d["mean"]),
            "std": float(d["std"]),
            "values": [float(v) for v in d["values"]],
        }
        for level, d in results.items()
    }
    with open(out_path, "w") as fp:
        json.dump(serial, fp, indent=2)
    print(f"  saved -> {out_path}")


def run_eval(module, net_class, perturbation, delay_tag, ckpt_token,
             eval_levels, key_fn):
    """Evaluate the ``perturbation``/``delay_tag`` checkpoint across all levels.

    Loads ``{perturbation}_{ds}_{delay_tag}_{ckpt_token}.pt`` for every
    dataset, evaluates it at each level in ``eval_levels`` and writes one
    ``{perturbation}_1stLayer_{ds}_{delay_tag}_evalon_{ckpt_token}.json`` per
    dataset to ``EVAL_LOG_DIR``.
    """
    use_delay = delay_tag == "delay"
    for dataset_key in EVAL_DATASETS:
        ckpt_file = (
            f"{perturbation}_{dataset_key}_{delay_tag}_{ckpt_token}.pt"
        )
        print(f"[{perturbation}/{delay_tag}] dataset={dataset_key} "
              f"| checkpoint={ckpt_file}")
        net = load_checkpoint(
            module, net_class, dataset_key, ckpt_file, use_delay
        )
        results = evaluate_across_levels(module, net, dataset_key, eval_levels)
        out_path = EVAL_LOG_DIR / (
            f"{perturbation}_1stLayer_{dataset_key}_{delay_tag}_"
            f"evalon_{ckpt_token}.json"
        )
        save_sweep_json(results, out_path, key_fn=key_fn)

## 1a. Jitter (per-spike) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_SIGMA_JITTER` across
the full jitter sweep (`jitter_mod.SIGMA_VALUES`).

In [21]:
# Jitter level of the checkpoint to evaluate (model trained at this sigma).
# Shared by the no-delay (1a) and delay (1b) sub-sections.
CHECKPOINT_SIGMA_JITTER = 10

run_eval(
    jitter_mod, jitter_mod.JitterSHDNetwork,
    perturbation="jitter", delay_tag="nodelay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_JITTER}",
    eval_levels=jitter_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[jitter/nodelay] dataset=whole | checkpoint=jitter_whole_nodelay_sigma10.pt
      eval@0: 0.2819 +/- 0.0000
      eval@1: 0.3022 +/- 0.0013
      eval@3: 0.4264 +/- 0.0023
      eval@5: 0.5462 +/- 0.0027
      eval@10: 0.6909 +/- 0.0057
      eval@17: 0.6386 +/- 0.0052
      eval@25: 0.5763 +/- 0.0060
  saved -> log_eval_on_perturbatedModel\jitter_1stLayer_whole_nodelay_evalon_sigma10.json
[jitter/nodelay] dataset=part | checkpoint=jitter_part_nodelay_sigma10.pt
      eval@0: 0.1795 +/- 0.0000
      eval@1: 0.2027 +/- 0.0050
      eval@3: 0.2589 +/- 0.0072
      eval@5: 0.3704 +/- 0.0042
      eval@10: 0.4912 +/- 0.0047
      eval@17: 0.4066 +/- 0.0046
      eval@25: 0.2625 +/- 0.0065
  saved -> log_eval_on_perturbatedModel\jitter_1stLayer_part_nodelay_evalon_sigma10.json
[jitter/nodelay] dataset=norm | checkpoint=jitter_norm_nodelay_sigma10.pt
      eval@0: 0.1306 +/- 0.0000
      eval@1: 0.1510 +/- 0.0038
      eval@3: 0.1697 +/- 0.0056
      eval@5: 0.2361 +/- 0.0070
      eval@10: 

## 1b. Jitter (per-spike) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_SIGMA_JITTER` across the full jitter sweep.

In [22]:
run_eval(
    jitter_mod, jitter_mod.JitterSHDNetwork,
    perturbation="jitter", delay_tag="delay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_JITTER}",
    eval_levels=jitter_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[jitter/delay] dataset=whole | checkpoint=jitter_whole_delay_sigma10.pt
      eval@0: 0.4890 +/- 0.0000
      eval@1: 0.5291 +/- 0.0045
      eval@3: 0.6600 +/- 0.0033
      eval@5: 0.7479 +/- 0.0013
      eval@10: 0.7813 +/- 0.0070
      eval@17: 0.7221 +/- 0.0066
      eval@25: 0.6462 +/- 0.0035
  saved -> log_eval_on_perturbatedModel\jitter_1stLayer_whole_delay_evalon_sigma10.json
[jitter/delay] dataset=part | checkpoint=jitter_part_delay_sigma10.pt
      eval@0: 0.2906 +/- 0.0000
      eval@1: 0.3118 +/- 0.0049
      eval@3: 0.3663 +/- 0.0089
      eval@5: 0.4884 +/- 0.0163
      eval@10: 0.5873 +/- 0.0035
      eval@17: 0.4375 +/- 0.0122
      eval@25: 0.3228 +/- 0.0035
  saved -> log_eval_on_perturbatedModel\jitter_1stLayer_part_delay_evalon_sigma10.json
[jitter/delay] dataset=norm | checkpoint=jitter_norm_delay_sigma10.pt
      eval@0: 0.1697 +/- 0.0000
      eval@1: 0.1945 +/- 0.0058
      eval@3: 0.2548 +/- 0.0063
      eval@5: 0.3138 +/- 0.0130
      eval@10: 0.3276 +/- 0.006

## 2a. Shift (per-neuron) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_SIGMA_SHIFT` across
the full shift sweep (`shift_mod.SIGMA_VALUES`).

In [23]:
# Shift level of the checkpoint to evaluate (model trained at this sigma).
# Shared by the no-delay (2a) and delay (2b) sub-sections.
CHECKPOINT_SIGMA_SHIFT = 10

run_eval(
    shift_mod, shift_mod.ShiftSHDNetwork,
    perturbation="shift", delay_tag="nodelay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_SHIFT}",
    eval_levels=shift_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[shift/nodelay] dataset=whole | checkpoint=shift_whole_nodelay_sigma10.pt
      eval@0: 0.2839 +/- 0.0000
      eval@1: 0.3013 +/- 0.0052
      eval@3: 0.3984 +/- 0.0030
      eval@5: 0.5222 +/- 0.0049
      eval@10: 0.6114 +/- 0.0063
      eval@17: 0.5084 +/- 0.0011
      eval@25: 0.3841 +/- 0.0028
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_whole_nodelay_evalon_sigma10.json
[shift/nodelay] dataset=part | checkpoint=shift_part_nodelay_sigma10.pt
      eval@0: 0.1978 +/- 0.0000
      eval@1: 0.1998 +/- 0.0012
      eval@3: 0.2446 +/- 0.0087
      eval@5: 0.3374 +/- 0.0061
      eval@10: 0.4094 +/- 0.0087
      eval@17: 0.2719 +/- 0.0063
      eval@25: 0.1461 +/- 0.0080
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_part_nodelay_evalon_sigma10.json
[shift/nodelay] dataset=norm | checkpoint=shift_norm_nodelay_sigma10.pt
      eval@0: 0.1087 +/- 0.0000
      eval@1: 0.1140 +/- 0.0012
      eval@3: 0.1498 +/- 0.0098
      eval@5: 0.1933 +/- 0.0066
      eval@10: 0.2267 +

## 2b. Shift (per-neuron) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_SIGMA_SHIFT` across the full shift sweep.

In [24]:
run_eval(
    shift_mod, shift_mod.ShiftSHDNetwork,
    perturbation="shift", delay_tag="delay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_SHIFT}",
    eval_levels=shift_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[shift/delay] dataset=whole | checkpoint=shift_whole_delay_sigma10.pt
      eval@0: 0.5331 +/- 0.0000
      eval@1: 0.5531 +/- 0.0045
      eval@3: 0.6457 +/- 0.0016
      eval@5: 0.7056 +/- 0.0063
      eval@10: 0.7188 +/- 0.0014
      eval@17: 0.6057 +/- 0.0120
      eval@25: 0.4778 +/- 0.0085
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_whole_delay_evalon_sigma10.json
[shift/delay] dataset=part | checkpoint=shift_part_delay_sigma10.pt
      eval@0: 0.3077 +/- 0.0000
      eval@1: 0.3260 +/- 0.0079
      eval@3: 0.3940 +/- 0.0050
      eval@5: 0.4660 +/- 0.0038
      eval@10: 0.5104 +/- 0.0087
      eval@17: 0.3337 +/- 0.0061
      eval@25: 0.2243 +/- 0.0049
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_part_delay_evalon_sigma10.json
[shift/delay] dataset=norm | checkpoint=shift_norm_delay_sigma10.pt
      eval@0: 0.1966 +/- 0.0000
      eval@1: 0.2092 +/- 0.0029
      eval@3: 0.2336 +/- 0.0042
      eval@5: 0.2723 +/- 0.0046
      eval@10: 0.2849 +/- 0.0060
      

## 3a. Deletion (per-spike) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_PD_DELETION` across
the full deletion sweep (`deletion_mod.PD_VALUES`).

In [25]:
# Deletion probability of the checkpoint to evaluate (model trained at this p_d).
# Shared by the no-delay (3a) and delay (3b) sub-sections.
CHECKPOINT_PD_DELETION = 0.6
_pd_token = f"pd{int(round(CHECKPOINT_PD_DELETION * 10)):02d}"

run_eval(
    deletion_mod, deletion_mod.DeletionSHDNetwork,
    perturbation="deletion", delay_tag="nodelay",
    ckpt_token=_pd_token,
    eval_levels=deletion_mod.PD_VALUES,
    key_fn=lambda lvl: str(float(lvl)),
)

[deletion/nodelay] dataset=whole | checkpoint=deletion_whole_nodelay_pd06.pt
      eval@0.0: 0.4943 +/- 0.0000
      eval@0.2: 0.5046 +/- 0.0069
      eval@0.4: 0.5146 +/- 0.0060
      eval@0.6: 0.5023 +/- 0.0072
      eval@0.8: 0.3605 +/- 0.0070
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_whole_nodelay_evalon_pd06.json
[deletion/nodelay] dataset=part | checkpoint=deletion_part_nodelay_pd06.pt
      eval@0.0: 0.3126 +/- 0.0000
      eval@0.2: 0.3460 +/- 0.0149
      eval@0.4: 0.3716 +/- 0.0094
      eval@0.6: 0.3614 +/- 0.0113
      eval@0.8: 0.2120 +/- 0.0136
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_part_nodelay_evalon_pd06.json
[deletion/nodelay] dataset=norm | checkpoint=deletion_norm_nodelay_pd06.pt
      eval@0.0: 0.2027 +/- 0.0000
      eval@0.2: 0.2051 +/- 0.0046
      eval@0.4: 0.2096 +/- 0.0051
      eval@0.6: 0.1954 +/- 0.0017
      eval@0.8: 0.1477 +/- 0.0046
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_norm_nodelay_evalon_pd06.jso

## 3b. Deletion (per-spike) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_PD_DELETION` across the full deletion sweep.

In [26]:
run_eval(
    deletion_mod, deletion_mod.DeletionSHDNetwork,
    perturbation="deletion", delay_tag="delay",
    ckpt_token=_pd_token,
    eval_levels=deletion_mod.PD_VALUES,
    key_fn=lambda lvl: str(float(lvl)),
)

[deletion/delay] dataset=whole | checkpoint=deletion_whole_delay_pd06.pt
      eval@0.0: 0.7535 +/- 0.0000
      eval@0.2: 0.7562 +/- 0.0066
      eval@0.4: 0.7666 +/- 0.0025
      eval@0.6: 0.7388 +/- 0.0059
      eval@0.8: 0.4578 +/- 0.0088
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_whole_delay_evalon_pd06.json
[deletion/delay] dataset=part | checkpoint=deletion_part_delay_pd06.pt
      eval@0.0: 0.6068 +/- 0.0000
      eval@0.2: 0.6186 +/- 0.0109
      eval@0.4: 0.6048 +/- 0.0106
      eval@0.6: 0.5653 +/- 0.0030
      eval@0.8: 0.3061 +/- 0.0072
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_part_delay_evalon_pd06.json
[deletion/delay] dataset=norm | checkpoint=deletion_norm_delay_pd06.pt
      eval@0.0: 0.3248 +/- 0.0000
      eval@0.2: 0.3545 +/- 0.0060
      eval@0.4: 0.3663 +/- 0.0026
      eval@0.6: 0.3284 +/- 0.0079
      eval@0.8: 0.1551 +/- 0.0036
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_norm_delay_evalon_pd06.json
